# 01 - MSFT: data, features, baselines

Prototype the whole pipeline here, then move finished code into `src/stockpred/`.
Rule: every number that ends up on the resume must come from output in this notebook
or from `outputs/metrics.csv`.

**Question:** can simple models predict MSFT's next-day direction better than trivial baselines,
after transaction costs?

In [9]:
# Imports and settings.
# Needed: numpy, pandas, yfinance, matplotlib, scikit-learn, xgboost.
# Set a global random seed so reruns match.

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import pandas_ta_classic as ta


## 1. Data

- Stock: MSFT. Reasons: liquid, long history, calmer than NVDA/TSLA, and different from the
  AAPL used in the KPI Anomaly Detection project. Note the survivorship bias (a hindsight pick).
- Download daily bars from 2010-01-01 with adjusted prices.
- Flatten yfinance's MultiIndex columns if present.
- Inspect: `.info()`, date range, missing days, NaNs, zero-volume days.
- Plot the close price. Any split-sized jumps mean the data is unadjusted.

In [10]:
# Download MSFT here
from tarfile import data_filter


stock = yf.Ticker("MSFT")

df = stock.history(start="2010-01-01")

df.head(10)

/Users/dhwanil/Desktop/ONGOING PROJECTS/stock market price prediction/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/dhwanil/Desktop/ONGOING PROJECTS/stock market price prediction/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/dhwanil/Desktop/ONGOING PROJECTS/stock market price prediction/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed i

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2010-01-04 00:00:00-05:00,22.739168,23.095627,22.716889,22.984234,38409100,0.0,0.0
2010-01-05 00:00:00-05:00,22.909971,23.095628,22.754019,22.991659,49749600,0.0,0.0
2010-01-06 00:00:00-05:00,22.932258,23.080784,22.664914,22.850571,58182400,0.0,0.0
2010-01-07 00:00:00-05:00,22.746594,22.798579,22.419840,22.612923,50559700,0.0,0.0
2010-01-08 00:00:00-05:00,22.486680,22.932254,22.456975,22.768877,51197400,0.0,0.0
2010-01-11 00:00:00-05:00,22.806012,22.843144,22.367864,22.479258,68754700,0.0,0.0
2010-01-12 00:00:00-05:00,22.390135,22.575791,22.211905,22.330725,65912100,0.0,0.0
2010-01-13 00:00:00-05:00,22.471824,22.664906,22.286168,22.538660,51863500,0.0,0.0
2010-01-14 00:00:00-05:00,22.508953,23.095628,22.471823,22.991659,63228100,0.0,0.0


In [11]:
# Data quality checks here

print(df.shape)

print(df.isnull().sum())

print(df.duplicated().sum())

print(df.describe())

print(df.info())

print(df.columns)


(4231, 7)
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64
0
              Open         High          Low        Close        Volume  \
count  4231.000000  4231.000000  4231.000000  4231.000000  4.231000e+03   
mean    162.783166   164.340000   161.170898   162.821197  3.520552e+07   
std     154.541067   155.968548   153.050263   154.556890  2.046484e+07   
min      17.305124    17.477500    17.035316    17.245171  5.855900e+06   
25%      32.212487    32.710521    31.867418    32.494556  2.207020e+07   
50%      90.467827    91.279378    89.637835    90.661499  2.940580e+07   
75%     277.291964   279.900014   274.080444   277.240860  4.286580e+07   
max     549.795172   551.150024   543.059998   543.450012  3.193179e+08   

         Dividends  Stock Splits  
count  4231.000000        4231.0  
mean      0.007343           0.0  
std       0.064638           0.0  
min       0.000000           0.0 

## 2. Features and target

Rule: a feature on row t uses data up to the close of day t only.

- Daily returns from the adjusted close.
- Keep the feature set small (8 to 12). With ~4,000 rows, more features means more overfitting.
- Features: lagged returns (1, 2, 5, 10, 21), rolling volatility (5, 21, 63), RSI(14),
  volume z-score, distance from 50- and 200-day moving averages.
- Target: 1 if the next day's return > 0, else 0. Keep `next_return` separately for the backtest.
- Drop warm-up rows (200-day window) and the last row.
- Check the class balance of the target (expect about 53% up).

In [12]:
# Build features and target here

# daily returns
df['daily_return'] = df['Close'].pct_change()

# rolling volatility
df['rolling_volatility'] = df['daily_return'].rolling(window=21).std()

# RSI
df['rsi'] = ta.rsi(df['Close'], length=14)

# distance from 50-day moving average
df['50_day_ma'] = df['Close'].rolling(window=50).mean()
df['200_day_ma'] = df['Close'].rolling(window=200).mean()

# distance from 200-day moving average
df['distance_50ma'] = df['Close'] - df['50_day_ma']
df['distance_200ma'] = df['Close'] - df['200_day_ma']

# volume z-score
df['volume_zscore'] = (df['Volume'] - df['Volume'].mean()) / df['Volume'].std()

## 3. Leakage check

Alter prices after some date t and confirm that the features on rows <= t do not change.
Then confirm the target is the only column that looks forward. Write this as a function,
because it becomes the first pytest test.

In [13]:
# Leakage check here

## 4. Baselines

Compute on the same test periods the models will use:
- Always-up (accuracy ~ the up-fraction, AUC = 0.5)
- Previous-day direction
- Buy-and-hold, with the same cost model

These set the bar. Write down the numbers. Buy-and-hold on MSFT since 2010 is strong
(survivorship), so a long/flat strategy that is out of the market part of the time faces a
high bar on raw return. Compare risk-adjusted numbers (Sharpe, drawdown) too.

In [14]:
# Baselines here

## 5. Walk-forward prototype

- Hold out an initial ~3-year training period, then 8 consecutive test blocks of ~6 months.
  Expanding training window, split by date, small gap between train and test.
- Logistic regression (scaler fit on the train window only), then XGBoost (shallow, regularised).
- Per fold: AUC and accuracy. A fold has only ~125 rows, so per-fold numbers are noisy.
  Judge on the pooled out-of-sample predictions.
- Strategy: long if P(up) > 0.5, else flat. Subtract 5 bps on each position change.
- Metrics: net Sharpe (sqrt(252)), max drawdown, equity curve.
- Sanity checks: a shuffled-target model should score AUC ~ 0.5, and an oracle should score 1.0.
- Bootstrap a confidence interval for pooled AUC and Sharpe.

In [15]:
# Fold construction here

In [16]:
# Fit, predict, score here

In [17]:
# Strategy returns, costs, equity curves here

In [18]:
# Bootstrap confidence intervals here

## 6. Findings

Fill in after running. Cover:
- Did either model beat the baselines in pooled AUC? By how much, and does the confidence interval exclude 0.5?
- What happened to the edge after costs, and versus buy-and-hold?
- Which features mattered (feature importances here, SHAP later)?
- What are the limits? (one stock picked with hindsight, one cost assumption, daily bars only, small sample)

If the honest result is "barely better than the baseline", say that. It is a defensible resume story.